# 9.2.长短期记忆网络（LSTM）

长期以来，隐变量模型存在着长期信息保存和短期输入缺失的问题。 解决这一问题的最早方法之一是长短期记忆网络（long short-term memory，LSTM） ([Hochreiter, 1997](https://zh.d2l.ai/chapter_references/zreferences.html#Hochreiter.Schmidhuber.1997))。 它有许多与门控循环单元（[9.1节](09.01_gru.ipynb)）一样的属性。 有趣的是，长短期记忆网络的设计比门控循环单元稍微复杂一些， 却比门控循环单元早诞生了近20年。

---
## 9.2.1.环境配置

In [1]:
%pip install pypto==0.2.0 torch torch_npu matplotlib

In [2]:
import os
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="On the interactive interface")
warnings.filterwarnings("ignore", message="Cannot create tensor")
import pypto
import torch
from torch import nn
from torch.nn import functional as F
import torch_npu
import logging
logging.getLogger('matplotlib').setLevel(logging.WARNING)
import matplotlib.pyplot as plt

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()

In [3]:
from src.utils import load_data_time_machine

batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
import torch
from torch import nn
from d2l import torch as d2l

batch_size, num_steps = 32, 35
train_iter, vocab = d2l.load_data_time_machine(batch_size, num_steps)
    </pre>
  </div>
</details>


---
## 9.2.2.门控记忆元

可以说，长短期记忆网络的设计灵感来自于计算机的逻辑门。 长短期记忆网络引入了*记忆元*（memory cell），或简称为*单元*（cell）。 有些文献认为记忆元是隐状态的一种特殊类型， 它们与隐状态具有相同的形状，其设计目的是用于记录附加的信息。 为了控制记忆元，我们需要许多门。 其中一个门用来从单元中输出条目，我们将其称为*输出门*（output gate）。 另外一个门用来决定何时将数据读入单元，我们将其称为*输入门*（input gate）。 我们还需要一种机制来重置单元的内容，由*遗忘门*（forget gate）来管理， 这种设计的动机与门控循环单元相同， 能够通过专用机制决定什么时候记忆或忽略隐状态中的输入。 让我们看看这在实践中是如何运作的。

### 9.2.2.1.输入门、遗忘门和输出门

就如在门控循环单元中一样， 当前时间步的输入和前一个时间步的隐状态 作为数据送入长短期记忆网络的门中， 如下图所示。 它们由三个具有sigmoid激活函数的全连接层处理， 以计算输入门、遗忘门和输出门的值。 因此，这三个门的值都在$(0, 1)$的范围内。

<div align="center">
  <img src="./images/lstm-0.svg" alt="图9.2.1 长短期记忆模型中的输入门、遗忘门和输出门" width="400">
  <br><small>图9.2.1 长短期记忆模型中的输入门、遗忘门和输出门</small>
</div>

我们来细化一下长短期记忆网络的数学表达。 假设有$h$个隐藏单元，批量大小为$n$，输入数为$d$。 因此，输入为$\mathbf{X}_t \in \mathbb{R}^{n \times d}$， 前一时间步的隐状态为$\mathbf{H}_{t-1} \in \mathbb{R}^{n \times h}$。 相应地，时间步$t$的门被定义如下： 输入门是$\mathbf{I}_t \in \mathbb{R}^{n \times h}$， 遗忘门是$\mathbf{F}_t \in \mathbb{R}^{n \times h}$， 输出门是$\mathbf{O}_t \in \mathbb{R}^{n \times h}$。 它们的计算方法如下：

$$
\tag{9.2.1}
\begin{aligned}
\mathbf{I}_t &= \sigma(\mathbf{X}_t \mathbf{W}_{xi} + \mathbf{H}_{t-1} \mathbf{W}_{hi} + \mathbf{b}_i),\\
\mathbf{F}_t &= \sigma(\mathbf{X}_t \mathbf{W}_{xf} + \mathbf{H}_{t-1} \mathbf{W}_{hf} + \mathbf{b}_f),\\
\mathbf{O}_t &= \sigma(\mathbf{X}_t \mathbf{W}_{xo} + \mathbf{H}_{t-1} \mathbf{W}_{ho} + \mathbf{b}_o),
\end{aligned}
$$

其中$\mathbf{W}_{xi}, \mathbf{W}_{xf}, \mathbf{W}_{xo} \in \mathbb{R}^{d \times h}$ 和$\mathbf{W}_{hi}, \mathbf{W}_{hf}, \mathbf{W}_{ho} \in \mathbb{R}^{h \times h}$是权重参数， $\mathbf{b}_i, \mathbf{b}_f, \mathbf{b}_o \in \mathbb{R}^{1 \times h}$是偏置参数。

### 9.2.2.2.候选记忆元

由于还没有指定各种门的操作，所以先介绍*候选记忆元*（candidate memory cell） $\tilde{\mathbf{C}}_t \in \mathbb{R}^{n \times h}$。 它的计算与上面描述的三个门的计算类似， 但是使用$\tanh$函数作为激活函数，函数的值范围为$(-1, 1)$。 下面导出在时间步$t$处的方程：

$$\tag{9.2.2}\tilde{\mathbf{C}}_t = \text{tanh}(\mathbf{X}_t \mathbf{W}_{xc} + \mathbf{H}_{t-1} \mathbf{W}_{hc} + \mathbf{b}_c),$$

其中$\mathbf{W}_{xc} \in \mathbb{R}^{d \times h}$和 $\mathbf{W}_{hc} \in \mathbb{R}^{h \times h}$是权重参数， $\mathbf{b}_c \in \mathbb{R}^{1 \times h}$是偏置参数。

候选记忆元的计算流程如下图所示。

<div align="center">
  <img src="./images/lstm-1.svg" alt="图9.2.2 长短期记忆模型中的候选记忆元" width="400">
  <br><small>图9.2.2 长短期记忆模型中的候选记忆元</small>
</div>

### 9.2.2.3.记忆元

在门控循环单元中，有一种机制来控制输入和遗忘（或跳过）。 类似地，在长短期记忆网络中，也有两个门用于这样的目的： 输入门$\mathbf{I}_t$控制采用多少来自$\tilde{\mathbf{C}}_t$的新数据， 而遗忘门$\mathbf{F}_t$控制保留多少过去的 记忆元$\mathbf{C}_{t-1} \in \mathbb{R}^{n \times h}$的内容。 使用按元素乘法，得出：

$$\tag{9.2.3}\mathbf{C}_t = \mathbf{F}_t \odot \mathbf{C}_{t-1} + \mathbf{I}_t \odot \tilde{\mathbf{C}}_t.$$

如果遗忘门始终为$1$且输入门始终为$0$， 则过去的记忆元$\mathbf{C}_{t-1}$ 将随时间被保存并传递到当前时间步。 引入这种设计是为了缓解梯度消失问题， 并更好地捕获序列中的长距离依赖关系。

这样我们就得到了计算记忆元的流程图，如下图所示。

<div align="center">
  <img src="./images/lstm-2.svg" alt="图9.2.3 在长短期记忆网络模型中计算记忆元" width="400">
  <br><small>图9.2.3 在长短期记忆网络模型中计算记忆元</small>
</div>

### 9.2.2.4.隐状态

最后，我们需要定义如何计算隐状态 $\mathbf{H}_t \in \mathbb{R}^{n \times h}$， 这就是输出门发挥作用的地方。 在长短期记忆网络中，它仅仅是记忆元的$\tanh$的门控版本。 这就确保了$\mathbf{H}_t$的值始终在区间$(-1, 1)$内：

$$\tag{9.2.4}\mathbf{H}_t = \mathbf{O}_t \odot \tanh(\mathbf{C}_t).$$

只要输出门接近$1$，我们就能够有效地将所有记忆信息传递给预测部分， 而对于输出门接近$0$，我们只保留记忆元内的所有信息，而不需要更新隐状态。下图提供了数据流的图形化演示。

<div align="center">
  <img src="./images/lstm-3.svg" alt="图9.2.4 在长短期记忆模型中计算隐状态" width="400">
  <br><small>图9.2.4 在长短期记忆模型中计算隐状态</small>
</div>

---
## 9.2.3.从零开始实现

现在，我们从零开始实现长短期记忆网络。 与 [8.5节](../08_pypto_recurrent_neural_networks/08.05_rnn_scratch.ipynb)中的实验相同， 我们首先加载时光机器数据集。

### 9.2.3.1.初始化模型参数

接下来，我们需要定义和初始化模型参数。 如前所述，超参数`num_hiddens`定义隐藏单元的数量。 我们按照标准差$0.01$的高斯分布初始化权重，并将偏置项设为$0$。

In [4]:
def get_lstm_params(vocab_size, num_hiddens, device):
    num_inputs = num_outputs = vocab_size

    def normal(shape):
        return torch.randn(size=shape, device=device)*0.01

    def three():
        return (normal((num_inputs, num_hiddens)),
                normal((num_hiddens, num_hiddens)),
                torch.zeros(num_hiddens, device=device))

    W_xi, W_hi, b_i = three()  # 输入门参数
    W_xf, W_hf, b_f = three()  # 遗忘门参数
    W_xo, W_ho, b_o = three()  # 输出门参数
    W_xc, W_hc, b_c = three()  # 候选记忆元参数
    # 输出层参数
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    # 附加梯度
    params = [W_xi, W_hi, b_i, W_xf, W_hf, b_f, W_xo, W_ho, b_o, W_xc, W_hc,
              b_c, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def get_lstm_params(vocab_size, num_hiddens, device):
    num_inputs = num_outputs = vocab_size

    def normal(shape):
        return torch.randn(size=shape, device=device)*0.01

    def three():
        return (normal((num_inputs, num_hiddens)),
                normal((num_hiddens, num_hiddens)),
                torch.zeros(num_hiddens, device=device))

    W_xi, W_hi, b_i = three()  # 输入门参数
    W_xf, W_hf, b_f = three()  # 遗忘门参数
    W_xo, W_ho, b_o = three()  # 输出门参数
    W_xc, W_hc, b_c = three()  # 候选记忆元参数
    # 输出层参数
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    # 附加梯度
    params = [W_xi, W_hi, b_i, W_xf, W_hf, b_f, W_xo, W_ho, b_o, W_xc, W_hc,
              b_c, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params
    </pre>
  </div>
</details>


### 9.2.3.2.定义模型

在**初始化函数**中， 长短期记忆网络的隐状态需要返回一个*额外*的记忆元， 单元的值为0，形状为（批量大小，隐藏单元数）。 因此，我们得到以下的状态初始化。

In [5]:
def init_lstm_state(batch_size, num_hiddens, device):
    return (torch.zeros((batch_size, num_hiddens), device=device),
            torch.zeros((batch_size, num_hiddens), device=device))

**实际模型**的定义与我们前面讨论的一样： 提供三个门和一个额外的记忆元。 请注意，只有隐状态才会传递到输出层， 而记忆元$\mathbf{C}_t$不直接参与输出计算。

---
## 9.2.4.LSTM 专用算子

LSTM 的四个门（输入门 I、遗忘门 F、输出门 O、候选记忆元 C̃）各需一次 `matmul(·)+matmul(·)+bias_add` 组合，再加上记忆元更新所需的逐元素 `mul` 和 `add`，总共涉及以下 PyPTO 算子：

<table style="margin-left: 0; margin-right: auto; width: auto; text-align: left;">
  <tr>
    <th style="padding: 8px; text-align: left;">算子</th>
    <th style="padding: 8px; text-align: left;">类名</th>
    <th style="padding: 8px; text-align: left;">用途</th>
    <th style="padding: 8px; text-align: left;">本节调用方式</th>
  </tr>
  <tr>
    <td style="padding: 8px;">矩阵乘法</td>
    <td style="padding: 8px;"><code>PyPTOMatmul</code></td>
    <td style="padding: 8px;"><code>X @ W</code>、<code>H @ W</code>、<code>H @ W_hq</code></td>
    <td style="padding: 8px;"><code>PyPTOMatmul.apply(...)</code></td>
  </tr>
  <tr>
    <td style="padding: 8px;">偏置加法</td>
    <td style="padding: 8px;"><code>PyPTOBiasAdd</code></td>
    <td style="padding: 8px;"><code>... + b</code> 广播</td>
    <td style="padding: 8px;"><code>PyPTOBiasAdd.apply(...)</code></td>
  </tr>
  <tr>
    <td style="padding: 8px;">逐元素加法</td>
    <td style="padding: 8px;"><code>PyPTOAdd</code></td>
    <td style="padding: 8px;">两矩阵相加</td>
    <td style="padding: 8px;"><code>PyPTOAdd.apply(...)</code></td>
  </tr>
  <tr>
    <td style="padding: 8px;">sigmoid</td>
    <td style="padding: 8px;"><code>PyPTOSigmoid</code></td>
    <td style="padding: 8px;">门激活 $\sigma$</td>
    <td style="padding: 8px;"><code>PyPTOSigmoid.apply(...)</code></td>
  </tr>
  <tr>
    <td style="padding: 8px;">tanh</td>
    <td style="padding: 8px;"><code>PyPTOTanh</code></td>
    <td style="padding: 8px;">候选元 / 输出激活</td>
    <td style="padding: 8px;"><code>PyPTOTanh.apply(...)</code>（独立 kernel）</td>
  </tr>
  <tr>
    <td style="padding: 8px;">逐元素乘法</td>
    <td style="padding: 8px;"><code>PyPTOMul</code></td>
    <td style="padding: 8px;">$F \odot C$、$I \odot \tilde{C}$ 等</td>
    <td style="padding: 8px;"><code>PyPTOMul.apply(...)</code></td>
  </tr>
</table>

上述算子均已收录在 `src/pypto_ops.py` 中（含前向 / 反向 kernel 与工厂函数），**本节直接从共享模块导入即可，无需内联 kernel 代码**。

In [6]:
from src.pypto_ops import PyPTOMatmul, PyPTOBiasAdd, PyPTOAdd
from src.pypto_ops import PyPTOTanh, PyPTOSigmoid, PyPTOMul
from src.pypto_ops import loss_fn

def lstm_pypto(inputs, state, params):
    """使用 PyPTO 算子在所有时间步上执行 LSTM 前向传播。"""
    [W_xi, W_hi, b_i, W_xf, W_hf, b_f, W_xo, W_ho, b_o, W_xc, W_hc, b_c,
     W_hq, b_q] = params
    (H, C) = state
    outputs = []
    for X in inputs:
        # 输入门 I
        xw_i = PyPTOMatmul.apply(X, W_xi)
        hw_i = PyPTOMatmul.apply(H, W_hi)
        i_in = PyPTOBiasAdd.apply(PyPTOAdd.apply(xw_i, hw_i), b_i)
        I = PyPTOSigmoid.apply(i_in)
        # 遗忘门 F
        xw_f = PyPTOMatmul.apply(X, W_xf)
        hw_f = PyPTOMatmul.apply(H, W_hf)
        f_in = PyPTOBiasAdd.apply(PyPTOAdd.apply(xw_f, hw_f), b_f)
        F = PyPTOSigmoid.apply(f_in)
        # 输出门 O
        xw_o = PyPTOMatmul.apply(X, W_xo)
        hw_o = PyPTOMatmul.apply(H, W_ho)
        o_in = PyPTOBiasAdd.apply(PyPTOAdd.apply(xw_o, hw_o), b_o)
        O = PyPTOSigmoid.apply(o_in)
        # 候选记忆元 C_tilde
        xw_c = PyPTOMatmul.apply(X, W_xc)
        hw_c = PyPTOMatmul.apply(H, W_hc)
        c_in = PyPTOBiasAdd.apply(PyPTOAdd.apply(xw_c, hw_c), b_c)
        C_tilde = PyPTOTanh.apply(c_in)
        # 记忆元更新: C = F * C + I * C_tilde
        f_C = PyPTOMul.apply(F, C)
        i_Ct = PyPTOMul.apply(I, C_tilde)
        C = PyPTOAdd.apply(f_C, i_Ct)
        # 隐状态: H = O * tanh(C)
        H = PyPTOMul.apply(O, PyPTOTanh.apply(C))
        # 输出层
        Y = PyPTOBiasAdd.apply(PyPTOMatmul.apply(H, W_hq), b_q)
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H, C)

<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f9f9fb; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <ul style="margin: 0; padding-left: 20px;">
      <li style="margin: 0 0 8px 0;"><b>LSTM 4 个门的计算模式高度统一</b>：每个门都是 <code>X@W_x + H@W_h + bias</code>，然后通过 sigmoid（I/F/O）或 tanh（C̃）激活。使用 <code>.apply()</code> 链路串联 PyPTO 算子后，前向在 NPU 上执行，反向自动调用对应的 backward kernel。</li>
      <li style="margin: 0 0 8px 0;"><b>与 8.5 节 RNN 的区别</b>：RNN 只有一对 W_xh/W_hh + tanh，而 LSTM 有 4 组权重参数（I/F/O/C̃），额外引入 sigmoid 门控和记忆元 C。逐元素 mul 和 add 算子首次在本章使用。</li>
      <li style="margin: 0 0 8px 0;"><b>PyPTOLSTM 模块</b>：在"简洁实现"小节，<code>src/pypto_ops</code> 中还提供了预封装的 <code>PyPTOLSTM(nn.Module)</code>，内部已完成参数注册与四门前向逻辑，可直接替换 <code>nn.LSTM</code>。</li>
      <li style="margin: 0 0 8px 0;"><b>loss_fn</b>：使用 PyPTO softmax + CE kernel，自动计算 log-softmax 和交叉熵损失。</li>
    </ul>
  </div>
</details>

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def init_lstm_state(batch_size, num_hiddens, device):
    return (torch.zeros((batch_size, num_hiddens), device=device),
            torch.zeros((batch_size, num_hiddens), device=device))
    </pre>
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def lstm(inputs, state, params):
    [W_xi, W_hi, b_i, W_xf, W_hf, b_f, W_xo, W_ho, b_o, W_xc, W_hc, b_c,
     W_hq, b_q] = params
    (H, C) = state
    outputs = []
    for X in inputs:
        I = torch.sigmoid((X @ W_xi) + (H @ W_hi) + b_i)
        F = torch.sigmoid((X @ W_xf) + (H @ W_hf) + b_f)
        O = torch.sigmoid((X @ W_xo) + (H @ W_ho) + b_o)
        C_tilde = torch.tanh((X @ W_xc) + (H @ W_hc) + b_c)
        C = F * C + I * C_tilde
        H = O * torch.tanh(C)
        Y = (H @ W_hq) + b_q
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H, C)
    </pre>
  </div>
</details>


接下来，我们**定义训练用的模型类**，封装参数初始化、状态初始化和前向传播：


In [7]:
class PyPTORNNModelScratch:
    """PyPTO 从零实现的 RNN 模型。

    LSTM 的隐状态包含 (H, C) 双元组，begin_state 返回
    init_lstm_state 的 (batch, hiddens) x2 张量。
    """
    def __init__(self, vocab_size, num_hiddens, device,
                 get_params_fn=None, init_state_fn=None, forward_fn=None):
        self.vocab_size = vocab_size
        self.num_hiddens = num_hiddens
        self.device = device
        self.params = (get_params_fn or get_lstm_params)(vocab_size, num_hiddens, device)
        self.init_state = init_state_fn or init_lstm_state
        self.forward_fn = forward_fn or lstm_pypto

    def __call__(self, X, state):
        X = torch.nn.functional.one_hot(X.T, self.vocab_size).type(torch.float32)
        return self.forward_fn(X, state, self.params)

    def begin_state(self, batch_size, device=None):
        if device is None:
            device = self.device
        return self.init_state(batch_size, self.num_hiddens, device)

In [8]:
num_hiddens = 512
net = PyPTORNNModelScratch(len(vocab), num_hiddens, device)
X = torch.arange(10).reshape((2, 5)).to(device)
state = net.begin_state(X.shape[0], device)
Y, new_state = net(X, state)
Y.shape, len(new_state), new_state[0].shape

(torch.Size([10, 28]), 2, torch.Size([2, 512]))

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
class RNNModelScratch:
    """从零开始实现的循环神经网络模型"""
    def __init__(self, vocab_size, num_hiddens, device,
                 get_params, init_state, forward_fn):
        self.vocab_size, self.num_hiddens = vocab_size, num_hiddens
        self.params = get_params(vocab_size, num_hiddens, device)
        self.init_state, self.forward_fn = init_state, forward_fn

    def __call__(self, X, state):
        X = F.one_hot(X.T, self.vocab_size).type(torch.float32)
        return self.forward_fn(X, state, self.params)

    def begin_state(self, batch_size, device):
        return self.init_state(batch_size, self.num_hiddens, device)
    </pre>
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
vocab_size, num_hiddens, device = len(vocab), 256, d2l.try_gpu()
num_epochs, lr = 500, 1
model = d2l.RNNModelScratch(len(vocab), num_hiddens, device, get_lstm_params,
                            init_lstm_state, lstm)
d2l.train_ch8(model, train_iter, vocab, lr, num_epochs, device)
    </pre>
  </div>
</details>


### 9.2.4.1.训练和预测

让我们通过实例化`PyPTORNNModelScratch`类（PyPTO 版 [8.5节](../08_pypto_recurrent_neural_networks/08.05_rnn_scratch.ipynb)中引入的`RNNModelScratch`）来训练一个长短期记忆网络， 就如我们在 [9.1节](09.01_gru.ipynb)中所做的一样。

In [9]:
def predict_ch8(prefix, num_preds, net, vocab, device):
    """在 prefix 后生成新字符。"""
    state = net.begin_state(batch_size=1, device=device)
    outputs = [vocab[prefix[0]]]
    get_input = lambda: torch.tensor([outputs[-1]], device=device).reshape((1, 1))
    for y in prefix[1:]: # 预热期
        _, state = net(get_input(), state)
        outputs.append(vocab[y])
    for _ in range(num_preds): # 预测num_preds步
        y, state = net(get_input(), state)
        outputs.append(int(y.argmax(dim=1).reshape(1)))
    return "".join([vocab.idx_to_token[i] for i in outputs])

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def predict_ch8(prefix, num_preds, net, vocab, device):
    """在prefix后面生成新字符"""
    state = net.begin_state(batch_size=1, device=device)
    outputs = [vocab[prefix[0]]]
    get_input = lambda: torch.tensor([outputs[-1]], device=device).reshape((1, 1))
    for y in prefix[1:]:  # 预热期
        _, state = net(get_input(), state)
        outputs.append(vocab[y])
    for _ in range(num_preds):  # 预测num_preds步
        y, state = net(get_input(), state)
        outputs.append(int(y.argmax(dim=1).reshape(1)))
    return ''.join([vocab.idx_to_token[i] for i in outputs])
    </pre>
  </div>
</details>


为了训练过程中的梯度稳定性，我们使用**梯度裁剪**：


In [10]:
def grad_clipping(net, theta):
    """裁剪梯度。"""
    if isinstance(net, nn.Module):
        params = [p for p in net.parameters() if p.requires_grad]
    else:
        params = net.params
    norm = torch.sqrt(sum(torch.sum((p.grad ** 2)) for p in params))
    if norm > theta:
        for param in params:
            param.grad[:] *= theta / norm

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def grad_clipping(net, theta):
    """裁剪梯度"""
    if isinstance(net, nn.Module):
        params = [p for p in net.parameters() if p.requires_grad]
    else:
        params = net.params
    norm = torch.sqrt(sum(torch.sum((p.grad ** 2)) for p in params))
    if norm &gt; theta:
        for param in params:
            param.grad[:] *= theta / norm
    </pre>
  </div>
</details>


---
## 9.2.5.训练

训练流程与 [8.5 节](../08_pypto_recurrent_neural_networks/08.05_rnn_scratch.ipynb)基本一致，但 LSTM 模型有三个特殊之处：
1. **双隐状态**：LSTM 的 `begin_state()` 返回 `(H, C)` 二元组，分别表示隐状态与记忆元
2. **标签展平**：RNN 输出 shape 为 `(batch*steps, vocab)`，标签需从 `(batch, steps)` 展平为 `(batch*steps,)`
3. **梯度裁剪**：更新参数前先裁剪梯度，防止长序列反向传播导致梯度爆炸

与 [9.1节](09.01_gru.ipynb) 一致，`train_epoch_ch8`、`train_ch8` 定义如下，其中 `sgd`、`Timer`、`Accumulator` 等辅助工具从 `src/utils.py` 导入。`loss.backward()` 时 PyTorch 沿计算图依次调用各 `autograd.Function.backward()`，触发对应的 PyPTO 反向 kernel。**首次运行**会触发编译（耗时较长），编译完成后直接复用缓存。

In [11]:
import math
from src.utils import Timer, Accumulator, sgd

def train_epoch_ch8(net, train_iter, loss, updater, device, use_random_iter):
    """训练网络一个迭代周期"""
    state, timer = None, Timer()
    metric = Accumulator(2)
    for X, Y in train_iter:
        if state is None or use_random_iter:
            state = net.begin_state(batch_size=X.shape[0], device=device)
        else:
            if isinstance(net, nn.Module) and not isinstance(state, tuple):
                state = state.detach()
            else:
                for s in state:
                    s.detach_()
        y = Y.T.reshape(-1)
        X, y = X.to(device), y.to(device)
        y_hat, state = net(X, state)
        l = loss(y_hat, y.long())
        if isinstance(updater, torch.optim.Optimizer):
            updater.zero_grad()
            l.backward()
            grad_clipping(net, 1)
            updater.step()
        else:
            l.backward()
            grad_clipping(net, 1)
            updater(batch_size=1)
        metric.add(l * y.numel(), y.numel())
    return math.exp(metric[0] / metric[1]), metric[1] / timer.stop()

def train_ch8(net, train_iter, vocab, lr, num_epochs, device,
              use_random_iter=False, use_plot=True, loss_fn=None):
    """训练 RNN
    Args:
        loss_fn: 可选，自定义损失函数（签名 loss(logits, labels) → 标量）
                 默认 nn.CrossEntropyLoss()
    """
    loss = loss_fn if loss_fn is not None else nn.CrossEntropyLoss()
    if isinstance(net, nn.Module):
        updater = torch.optim.SGD(net.parameters(), lr)
    else:
        updater = lambda batch_size: sgd(net.params, lr, batch_size)
    predict = lambda prefix: predict_ch8(prefix, 50, net, vocab, device)
    for epoch in range(num_epochs):
        ppl, speed = train_epoch_ch8(
            net, train_iter, loss, updater, device, use_random_iter)
    print(f"困惑度 {ppl:.1f}, {speed:.1f} 词元/秒 {str(device)}")
    print(predict("time traveller"))
    print(predict("traveller"))

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def train_epoch_ch8(net, train_iter, loss, updater, device, use_random_iter):
    """训练网络一个迭代周期（定义见第8章）"""
    state, timer = None, d2l.Timer()
    metric = d2l.Accumulator(2)  # 训练损失之和,词元数量
    for X, Y in train_iter:
        if state is None or use_random_iter:
            state = net.begin_state(batch_size=X.shape[0], device=device)
        else:
            if isinstance(net, nn.Module) and not isinstance(state, tuple):
                state.detach_()
            else:
                for s in state:
                    s.detach_()
        y = Y.T.reshape(-1)
        X, y = X.to(device), y.to(device)
        y_hat, state = net(X, state)
        l = loss(y_hat, y.long()).mean()
        if isinstance(updater, torch.optim.Optimizer):
            updater.zero_grad()
            l.backward()
            grad_clipping(net, 1)
            updater.step()
        else:
            l.backward()
            grad_clipping(net, 1)
            updater(batch_size=1)
        metric.add(l * y.numel(), y.numel())
    return math.exp(metric[0] / metric[1]), metric[1] / timer.stop()
    </pre>
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def train_ch8(net, train_iter, vocab, lr, num_epochs, device,
              use_random_iter=False):
    """训练模型（定义见第8章）"""
    loss = nn.CrossEntropyLoss()
    animator = d2l.Animator(xlabel='epoch', ylabel='perplexity',
                            legend=['train'], xlim=[10, num_epochs])
    if isinstance(net, nn.Module):
        updater = torch.optim.SGD(net.parameters(), lr)
    else:
        updater = lambda batch_size: d2l.sgd(net.params, lr, batch_size)
    predict = lambda prefix: predict_ch8(prefix, 50, net, vocab, device)
    for epoch in range(num_epochs):
        ppl, speed = train_epoch_ch8(
            net, train_iter, loss, updater, device, use_random_iter)
        if (epoch + 1) % 10 == 0:
            print(predict('time traveller'))
            animator.add(epoch + 1, [ppl])
    print(f'困惑度 {ppl:.1f}, {speed:.1f} 词元/秒 {str(device)}')
    print(predict('time traveller'))
    print(predict('traveller'))
    </pre>
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
vocab_size, num_hiddens, device = len(vocab), 256, d2l.try_gpu()
num_epochs, lr = 500, 1
model = d2l.RNNModelScratch(len(vocab), num_hiddens, device, get_lstm_params,
                            init_lstm_state, lstm)
d2l.train_ch8(model, train_iter, vocab, lr, num_epochs, device)
    </pre>
  </div>
</details>


### 9.2.5.1.PyPTO 训练说明

`loss.backward()` 时 PyTorch 沿计算图依次调用各 `autograd.Function.backward()`，触发对应的 PyPTO 反向 kernel。**首次运行**会触发编译（耗时较长），编译完成后直接复用缓存。下面的 warmup 单元会预先触发 JIT 编译，避免训练过程中首次 batch 等待。

In [12]:
# 预热：触发所有 PyPTO kernel 的首次编译
print('正在编译 pypto kernel（首次运行耗时较长，请耐心等待）...')
# 参数已在 get_lstm_params 中按 device 创建于 NPU 上，此处仅确保一致性
for p in net.params:
    p.data = p.data.to(device)
# 取一个批次，构造完整前向+反向计算来触发 JIT 编译
X, y = next(iter(train_iter))
X, y = X.to(device), y.to(device)
state = net.begin_state(X.shape[0], device)  # LSTM 需要初始 (H, C)
y_hat, state = net(X, state)
y = y.T.reshape(-1).to(device)               # 标签展平为 (batch*steps,)
l = loss_fn(y_hat, y.long(), num_classes=len(vocab))
l.backward()
# 重置梯度，为正式训练做准备
for p in net.params:
    if p.grad is not None:
        p.grad = None
print('编译完成（耗时较长）。接下来可以正常训练了。')

In [13]:
num_epochs, lr = 500, 1
train_ch8(net, train_iter, vocab, lr, num_epochs, device, use_plot=False,
          loss_fn=lambda y_hat, y: loss_fn(y_hat, y, num_classes=len(vocab)))

困惑度 1.1, 3599.4 词元/秒 npu:0


time traveller for so it will be convenient to speak of himwas e


travelleryou can show black is white by argument said filby


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
vocab_size, num_hiddens, device = len(vocab), 256, d2l.try_gpu()
num_epochs, lr = 500, 1
model = d2l.RNNModelScratch(len(vocab), num_hiddens, device, get_lstm_params,
                            init_lstm_state, lstm)
d2l.train_ch8(model, train_iter, vocab, lr, num_epochs, device)
    </pre>
  </div>
</details>


LSTM 从零实现训练 500 轮后**困惑度降至 1.1**，与 [9.1 节](09.01_gru.ipynb) GRU（1.0）基本相当。预测文本中 `himwas e` 这类无空格拼接是字符级语言模型的正常现象（字符间本来就无空格）。

---
## 9.2.6.简洁实现

使用高级API，我们可以直接实例化`LSTM`模型。 高级API封装了前文介绍的所有配置细节。 这段代码的运行速度要快得多， 因为它使用的是编译好的运算符而不是Python来处理之前阐述的许多细节。

In [14]:
from src.pypto_ops import PyPTOLSTM

num_inputs = len(vocab)
num_hiddens = 256  # 简洁实现与原书一致使用 256（从零实现为演示效果使用 512）

# 定义 LSTM 简洁模型：将 PyPTOLSTM 与输出线性层组合
class PyPTOLSTMModel(nn.Module):
    """PyPTO LSTM 简洁模型（nn.Module 版本）。"""
    def __init__(self, vocab_size, num_hiddens):
        super().__init__()
        self.vocab_size = vocab_size
        self.num_hiddens = num_hiddens
        self.lstm = PyPTOLSTM(vocab_size, num_hiddens)
        self.linear = nn.Linear(num_hiddens, vocab_size)

    def forward(self, X, state):
        X = F.one_hot(X.T.long(), self.vocab_size).type(torch.float32)
        Y, state = self.lstm(X, state)
        output = self.linear(Y.reshape((-1, Y.shape[-1])))
        return output, state

    def begin_state(self, batch_size, device):
        return (torch.zeros((self.lstm.num_layers, batch_size, self.num_hiddens), device=device),
                torch.zeros((self.lstm.num_layers, batch_size, self.num_hiddens), device=device))

model = PyPTOLSTMModel(len(vocab), num_hiddens)
model = model.to(device)
train_ch8(model, train_iter, vocab, lr, num_epochs, device, use_plot=False,
          loss_fn=lambda y_hat, y: loss_fn(y_hat, y, num_classes=len(vocab)))

困惑度 1.1, 4365.8 词元/秒 npu:0


time traveller for so it will be convenient to speak of himwas e


traveller with a slight accession ofcheerfulness really thi


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
num_inputs = vocab_size
lstm_layer = nn.LSTM(num_inputs, num_hiddens)
model = d2l.RNNModel(lstm_layer, len(vocab))
model = model.to(device)
d2l.train_ch8(model, train_iter, vocab, lr, num_epochs, device)
    </pre>
  </div>
</details>


简洁实现（基于 `PyPTOLSTM` 算子）**困惑度同样为 1.1**，速度约 4365 词元/秒。LSTM 与 GRU 在本数据集上表现接近，说明两者门控机制在字符级语言建模任务上能力相当。

长短期记忆网络是典型的具有重要状态控制的隐变量自回归模型。 多年来已经提出了其许多变体，例如，多层、残差连接、不同类型的正则化。 然而，由于序列的长距离依赖性，训练长短期记忆网络和其他序列模型（例如门控循环单元）的成本是相当高的。 在后面的内容中，我们将讲述更高级的替代模型，如Transformer。

---
## 9.2.7.小结

* 长短期记忆网络有三种类型的门：输入门、遗忘门和输出门。
* 长短期记忆网络的隐藏层输出包括"隐状态"和"记忆元"。只有隐状态会传递到输出层，而记忆元完全属于内部信息。
* 长短期记忆网络可以缓解梯度消失和梯度爆炸。

---
## 9.2.8.练习

1. 调整和分析超参数对运行时间、困惑度和输出顺序的影响。
1. 如何更改模型以生成适当的单词，而不是字符序列？
1. 在给定隐藏层维度的情况下，比较门控循环单元、长短期记忆网络和常规循环神经网络的计算成本。要特别注意训练和推断成本。
1. 既然候选记忆元通过使用$\tanh$函数来确保值范围在$(-1,1)$之间，那么为什么隐状态需要再次使用$\tanh$函数来确保输出值范围在$(-1,1)$之间呢？
1. 实现一个能够基于时间序列进行预测而不是基于字符序列进行预测的长短期记忆网络模型。

参考答案详见 [answers/09.02_reference_answer](./answers/09.02_reference_answer.ipynb)。

### 9.2.8.1.参考答案（PyPTO）

In [15]:
!cat answers/txt/09.02_reference_answer_pypto.txt

### 9.2.8.2.参考答案（PyTorch）


In [16]:
!cat answers/txt/09.02_reference_answer_pytorch.txt